In [1]:
import numpy as np
from metavision_core.event_io.raw_reader import RawReader
from matplotlib import pyplot as plt
from scipy import sparse
from tqdm import tqdm
import os
import gc
import cv2
from skimage.transform import warp
from skimage.registration import optical_flow_tvl1, optical_flow_ilk
from skimage.registration import phase_cross_correlation
from parallel_merge import process_line_by_line, process_line_by_line_45deg

In [2]:
"""
BLOCK: Read events from trigger mode
"""
file_name = ""
raw_stream_x = RawReader("../data/grayscale_example/"+file_name+"x.raw", max_events=int(5e10))
output_folder = "../results/grayscale_example/"
os.makedirs(output_folder, exist_ok=True)
events_x = raw_stream_x.load_n_events(int(4e10-1e6))
external_triggers_x = raw_stream_x.get_ext_trigger_events()
positive_triggers_x = external_triggers_x[external_triggers_x['p']==1]

print("Number of events loaded:", len(events_x))
print(f"Total number of external triggers: {len(external_triggers_x)}")
print(f"Number of positive external triggers: {len(positive_triggers_x)}")
print("First 4 external triggers:", external_triggers_x[:4])
print("First 2 positive external triggers:", positive_triggers_x[:2])

trigger_times = np.array([trigger['t'] for trigger in positive_triggers_x[:25]])
trigger_differences = np.diff(trigger_times)
print("Differences between consecutive triggers:", trigger_differences)
print("Time for one line scan:", trigger_times[-1]-trigger_times[0])

pixel_per_mm = int(5910)
total_scan_area = 7 #mm
num_trigger_per_mm = 10
num_trigger_per_line = int((total_scan_area - 6) * num_trigger_per_mm + 1) #number of external triggers per line
mm_per_move = 0.1 #mm
sensor_width = 1000 #pixel
overlap_width = int(sensor_width - mm_per_move * pixel_per_mm)
fully_reconstruction_area = int(total_scan_area-6) #mm
# merged_height = pixel_per_mm*total_scan_area + 360*2
merged_width = pixel_per_mm*total_scan_area + 640*2
trigger_interval = fully_reconstruction_area/(num_trigger_per_line-1) #mm
print(f"trigger_interval: {trigger_interval}")
print(f"overlap_width: {overlap_width}")

assert len(positive_triggers_x) == int(num_trigger_per_line * (fully_reconstruction_area / mm_per_move))
assert overlap_width > 50

lines_x = process_line_by_line(
    positive_triggers_t=positive_triggers_x['t'],
    events_t=events_x['t'],
    # events_x_coord=events_x['x'], # x.raw
    # events_y_coord=events_x['y'], # x.raw
    
    events_x_coord=events_x['y'], # y.raw
    events_y_coord=events_x['x'] - np.min(events_x['x']), # y.raw

    events_p=events_x['p'],
    total_lines=int(fully_reconstruction_area / mm_per_move),
    num_trigger_per_line=num_trigger_per_line,
    mm_per_move=mm_per_move,
    pixel_per_mm=pixel_per_mm,
    trigger_interval_x=trigger_interval,
    merged_height=merged_width,
    merged_width=sensor_width, 
    manual_shift=0,
    axis="y") 


Number of events loaded: 1207486040
Total number of external triggers: 220
Number of positive external triggers: 110
First 4 external triggers: [(0, 3137840, 0) (1, 3138081, 0) (0, 3157582, 0) (1, 3157844, 0)]
First 2 positive external triggers: [(1, 3138081, 0) (1, 3157844, 0)]
Differences between consecutive triggers: [  19763   19489   20250   20247   19751   20246   19997   20002   19756
   20247 1789699   20244   19998   19749   20249   19507   20489   19505
   20495   19502   20249 1823199   19994   19749]
Time for one line scan: 4052376
trigger_interval: 0.1
overlap_width: 409


In [3]:
# average corse shift calculation
average_shift_odd = 0
average_shift_even = 0
average_overlap_width = 0
count_invalid_odd = 0
count_invalid_even = 0
for i in tqdm(range(0, len(lines_x)-1, 1)):
    line_x1 = lines_x[i][:, -overlap_width-1:-1].copy()
    line_x2 = lines_x[i+1][:, 0:overlap_width].copy()
    line_x1 = np.where(np.abs(line_x1) > 2, line_x1, 0)
    line_x2 = np.where(np.abs(line_x2) > 2, line_x2, 0)
    shift, error, diffphase = phase_cross_correlation(line_x1, line_x2, reference_mask=line_x1 != 0, moving_mask=line_x2 != 0)
    print(f"shift: {shift}")
    if np.abs(shift[0]) > 300:
        if i % 2 == 0:
            count_invalid_odd += 1
        else:
            count_invalid_even += 1
        continue
    if i % 2 == 0:
        average_shift_odd += shift[0]
    else:
        average_shift_even += shift[0]
    average_overlap_width += shift[1]
average_shift_odd /= len(lines_x) // 2 - count_invalid_odd  
average_shift_even /= len(lines_x) // 2 - count_invalid_even
average_overlap_width /= len(lines_x) - count_invalid_odd - count_invalid_even
overlap_width = int(overlap_width - average_overlap_width)
print(f"average_shift_odd: {average_shift_odd}")
print(f"average_shift_even: {average_shift_even}")
print(f"average_overlap_width: {average_overlap_width}")
print(f"overlap_width: {overlap_width}")
print(f"count_invalid_odd: {count_invalid_odd}")
print(f"count_invalid_even: {count_invalid_even}")
assert overlap_width > 30

 11%|█         | 1/9 [00:02<00:19,  2.41s/it]

shift: [-20.   4.]


 22%|██▏       | 2/9 [00:04<00:16,  2.40s/it]

shift: [27. -3.]


 33%|███▎      | 3/9 [00:07<00:14,  2.40s/it]

...

shift: [-24.   4.]


 89%|████████▉ | 8/9 [00:19<00:02,  2.40s/it]

shift: [24. -4.]


100%|██████████| 9/9 [00:21<00:00,  2.40s/it]

shift: [-24.   4.]
average_shift_odd: -22.8
average_shift_even: 19.6
average_overlap_width: 0.3
overlap_width: 408
count_invalid_odd: 0
count_invalid_even: 0


In [4]:
for i in tqdm(range(0, len(lines_x)-1, 2)):
    lines_x[i+1] = np.roll(lines_x[i+1], int(average_shift_odd), axis=0)

100%|██████████| 5/5 [00:00<00:00, 1315.82it/s]


In [5]:
# DEBUG: Plot all event lines
line_width_x = sensor_width

empty_events = np.zeros((int(merged_width - 6*pixel_per_mm), line_width_x*int(fully_reconstruction_area / mm_per_move)), dtype=np.int8)
for i, line_x in enumerate(lines_x):
    empty_events[:, line_width_x * i : line_width_x * (i + 1)] = line_x

# np.save(os.path.join(output_folder, f"all_events.npy"), empty_events)
plt.imsave(os.path.join(output_folder, f"all_events_x.png"), empty_events[::8, ::8], vmin=-5, vmax=5)
plt.imsave(os.path.join(output_folder, f"exapmle_events_x.png"), lines_x[5], vmin=-5, vmax=5)

In [6]:
del events_x
del raw_stream_x
del empty_events
gc.collect()

48008

In [ ]:
merged_events_x = lines_x[0]

for i in tqdm(range(int(fully_reconstruction_area / mm_per_move - 1)), desc="Merging events"):
    # line_index_1 = i
    line_index_2 = i + 1

    line_x1 = merged_events_x[:, -overlap_width-1:-1].copy()
    line_x2 = lines_x[line_index_2][:, 0:overlap_width].copy()
    line_x1 = np.where(np.abs(line_x1) > 2, line_x1, 0)
    line_x2 = np.where(np.abs(line_x2) > 2, line_x2, 0)

    line_x1 = np.clip(line_x1, -8, 8)
    line_x2 = np.clip(line_x2, -8, 8)

    shift = [0, 0]
    shifted_line_x2 = line_x2
    shifted_x2 = lines_x[line_index_2]

    line_x1 = line_x1 - 9
    shifted_line_x2 = shifted_line_x2 - 9
    line_x1 = np.where(line_x1 != -9, line_x1, 0)
    shifted_line_x2 = np.where(shifted_line_x2 != -9, shifted_line_x2, 0)
    line_x1 = line_x1.astype(np.uint8)
    shifted_line_x2 = shifted_line_x2.astype(np.uint8)

    shift, error, diffphase = phase_cross_correlation(line_x1, shifted_line_x2, upsample_factor=10, reference_mask=line_x1 != 0, moving_mask=shifted_line_x2 != 0)
    if np.abs(shift[0]) > 50:
        print(f"unreasonable shift_x: {shift}, resetting to 0")
        shift[0] = 0
    if np.abs(shift[1]) > 20:
        print(f"unreasonable shift_y: {shift}, resetting to 0")
        shift[1] = 0

    initial_flow = np.full((line_x1.shape[0], line_x1.shape[1], 2), [-shift[1], -shift[0]], dtype=np.float32)
    flow = cv2.calcOpticalFlowFarneback(
        line_x1, shifted_line_x2, initial_flow,
        pyr_scale=0.95, levels=60, winsize=50,
        iterations=32, poly_n=7, poly_sigma=1.5, flags=cv2.OPTFLOW_USE_INITIAL_FLOW
    )
    u = flow[:, :, 0]
    v = flow[:, :, 1]
    avg_u = np.mean(u)
    y_shift = int(overlap_width + avg_u)
    
    # v, u = optical_flow_ilk(line_x1, shifted_line_x2, radius=400, num_warp=8)
    shifted_x2 = shifted_x2[:, y_shift:]
    nr, nc = shifted_x2.shape
    row_coords, col_coords = np.meshgrid(np.arange(nr), np.arange(nc), indexing='ij')

    v_per_row = np.mean(v, axis=1)
    x = np.linspace(0, 1, nc-overlap_width)
    x = np.concatenate((x, np.ones(overlap_width)))
    tqdm.write(f"shift: {shift}, avg_u: {avg_u}, min: {v_per_row.min()}, max: {v_per_row.max()}, line_max: {lines_x[line_index_2].max()}")
    v_per_row = v_per_row[:, np.newaxis] * (1 - x)

    row_coords, col_coords = np.meshgrid(np.arange(shifted_x2.shape[0]), np.arange(shifted_x2.shape[1]), indexing='ij')
    warp_shifted_x2 = warp(shifted_x2, np.array([row_coords + v_per_row, col_coords]), order=0)
    print(f"warp_shifted_x2 max: {warp_shifted_x2.max()}, min: {warp_shifted_x2.min()}")
    print(f"merged_events_x max: {merged_events_x.max()}, min: {merged_events_x.min()}")


    merged_events_x = np.concatenate((merged_events_x, warp_shifted_x2), axis=1)

np.save(os.path.join(output_folder, f"merged_shifted_final_x.npy"), merged_events_x)
plt.imsave(os.path.join(output_folder, f"merged_shifted_final_x.png"), merged_events_x, vmin=-10, vmax=10)
print(merged_events_x.shape)

Merging events:  11%|█         | 1/9 [00:25<03:24, 25.53s/it]

shift: [1. 4.], avg_u: -3.1011149883270264, min: -11.078444480895996, max: 8.973929405212402, line_max: 28
warp_shifted_x2 max: 28, min: -32
merged_events_x max: 31, min: -27


Merging events:  22%|██▏       | 2/9 [00:50<02:58, 25.48s/it]

shift: [ 4. -4.], avg_u: 3.7254629135131836, min: -13.042865753173828, max: 5.094882011413574, line_max: 31
warp_shifted_x2 max: 31, min: -26
merged_events_x max: 31, min: -32


Merging events:  33%|███▎      | 3/9 [01:16<02:32, 25.42s/it]

...

shift: [-2.  3.], avg_u: -3.46066951751709, min: -8.25662899017334, max: 12.00650405883789, line_max: 27
warp_shifted_x2 max: 27, min: -31
merged_events_x max: 31, min: -32


Merging events:  89%|████████▉ | 8/9 [03:23<00:25, 25.46s/it]

shift: [ 2. -5.], avg_u: 4.512112140655518, min: -11.421707153320312, max: 5.469123363494873, line_max: 30
warp_shifted_x2 max: 28, min: -27
merged_events_x max: 31, min: -32


Merging events: 100%|██████████| 9/9 [03:49<00:00, 25.45s/it]


shift: [-2.  3.], avg_u: -2.500938892364502, min: -4.761756896972656, max: 9.762207984924316, line_max: 27
warp_shifted_x2 max: 27, min: -34
merged_events_x max: 31, min: -32
(7190, 6330)
